# 07 — QLoRA Fine-Tuning

Fine-tune Qwen2.5-VL-7B-Instruct on our schema.org training dataset using QLoRA.

**Run this notebook on RunPod** — not locally. Requirements:
- GPU: 1× A100 80GB (SXM preferred; PCIe works)
- Template: RunPod PyTorch 2.1+ (CUDA 12.1)
- Setup: run `bash /workspace/schema-model/scripts/setup_runpod.sh` first

**Training details**:
- Base model: Qwen2.5-VL-7B-Instruct (quantized 4-bit, ~5GB)
- LoRA adapters: r=64, alpha=128, applied to attention + MLP layers
- Effective batch size: 16 (4 per device × 4 gradient accumulation steps)
- Duration: ~4-8 hours on A100 for 10K-50K examples
- Cost: ~$11-22 at RunPod Secure rates

In [ ]:
import sys
sys.path.insert(0, '/workspace/schema-model')  # RunPod path

import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('/workspace/schema-model/.env', override=True)

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DATA_DIR = Path('/workspace/schema-model/data/processed')
MODELS_DIR = Path('/workspace/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## Load Config and Data

In [ ]:
import yaml

with open('/workspace/schema-model/configs/training_config.yaml') as f:
    config = yaml.safe_load(f)

MODEL_ID = config['model']['name']
HF_TOKEN = os.getenv('HF_TOKEN')
OUTPUT_DIR = MODELS_DIR / config['training']['output_dir'].split('/')[-1]

print(f'Base model: {MODEL_ID}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Training data: {DATA_DIR}/train.jsonl')

# Check data exists
train_path = DATA_DIR / 'train.jsonl'
eval_path = DATA_DIR / 'eval.jsonl'
assert train_path.exists(), f'Missing training data at {train_path}'
assert eval_path.exists(), f'Missing eval data at {eval_path}'

# Count examples
n_train = sum(1 for _ in open(train_path))
n_eval = sum(1 for _ in open(eval_path))
print(f'Training examples: {n_train}')
print(f'Eval examples: {n_eval}')

## Load Model with 4-bit Quantization

In [ ]:
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
)
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f'Loading {MODEL_ID}...')
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=HF_TOKEN,
    cache_dir=str(MODELS_DIR),
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    cache_dir=str(MODELS_DIR),
)

print(f'Model loaded. Parameters: {model.num_parameters()/1e9:.2f}B')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M')

## Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for training (handles gradient checkpointing with 4-bit)
model = prepare_model_for_kbit_training(model)

lora_cfg = config['lora']
lora_config = LoraConfig(
    r=lora_cfg['r'],
    lora_alpha=lora_cfg['alpha'],
    lora_dropout=lora_cfg['dropout'],
    bias=lora_cfg['bias'],
    task_type=lora_cfg['task_type'],
    target_modules=lora_cfg['target_modules'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Prepare Dataset

In [ ]:
from datasets import load_dataset
from qwen_vl_utils import process_vision_info

# Load JSONL datasets
train_dataset = load_dataset('json', data_files=str(train_path), split='train')
eval_dataset = load_dataset('json', data_files=str(eval_path), split='train')

print(f'Train dataset: {len(train_dataset)} examples')
print(f'Eval dataset: {len(eval_dataset)} examples')

def preprocess_example(example):
    """Apply processor to convert messages into model inputs."""
    messages = example['messages']
    
    # Apply chat template
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    
    # Extract image inputs if present
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs if image_inputs else None,
        padding=True,
        truncation=True,
        max_length=config['training']['max_seq_length'],
        return_tensors='pt',
    )
    
    # Labels: mask padding tokens
    inputs['labels'] = inputs['input_ids'].clone()
    inputs['labels'][inputs['labels'] == processor.tokenizer.pad_token_id] = -100
    
    return {k: v.squeeze(0) for k, v in inputs.items()}

print('Preprocessing dataset...')
train_dataset = train_dataset.map(preprocess_example, batched=False, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(preprocess_example, batched=False, remove_columns=eval_dataset.column_names)
print('Preprocessing complete')

## Training

In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer

train_cfg = config['training']

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=train_cfg['epochs'],
    per_device_train_batch_size=train_cfg['per_device_train_batch_size'],
    gradient_accumulation_steps=train_cfg['gradient_accumulation_steps'],
    learning_rate=train_cfg['learning_rate'],
    lr_scheduler_type=train_cfg['lr_scheduler_type'],
    warmup_ratio=train_cfg['warmup_ratio'],
    bf16=train_cfg['bf16'],
    gradient_checkpointing=train_cfg['gradient_checkpointing'],
    logging_steps=train_cfg['logging_steps'],
    save_steps=train_cfg['save_steps'],
    eval_steps=train_cfg['eval_steps'],
    eval_strategy='steps',
    save_total_limit=train_cfg['save_total_limit'],
    load_best_model_at_end=train_cfg['load_best_model_at_end'],
    metric_for_best_model=train_cfg['metric_for_best_model'],
    report_to=train_cfg.get('report_to', 'none'),
    dataloader_num_workers=train_cfg.get('dataloader_num_workers', 4),
    remove_unused_columns=False,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=processor.tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print('Training configuration:')
print(f'  Effective batch size: {train_cfg["per_device_train_batch_size"] * train_cfg["gradient_accumulation_steps"]}')
print(f'  Learning rate: {train_cfg["learning_rate"]}')
print(f'  Epochs: {train_cfg["epochs"]}')

In [ ]:
# Start training
print('Starting fine-tuning...')
trainer.train()

# Save final model
trainer.save_model(str(OUTPUT_DIR / 'final'))
processor.save_pretrained(str(OUTPUT_DIR / 'final'))

print(f'\nTraining complete. Model saved to {OUTPUT_DIR / "final"}')

In [ ]:
# Training loss curve
import matplotlib.pyplot as plt
import pandas as pd

log_history = trainer.state.log_history
train_logs = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_logs = [l for l in log_history if 'eval_loss' in l]

if train_logs:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot([l['step'] for l in train_logs], [l['loss'] for l in train_logs], label='Train loss')
    if eval_logs:
        ax.plot([l['step'] for l in eval_logs], [l['eval_loss'] for l in eval_logs], label='Eval loss')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('Fine-tuning Loss')
    ax.legend()
    plt.tight_layout()
    plt.show()

print('Next step: 08_evaluation.ipynb')